In [8]:
import jax
import jax.numpy as jnp
import jax.random as jar
from flax import struct

import malthusjax as mjx
from malthusjax.engine import GeneticEngine, GeneticEngineParams, DiversityAwareEngine
from malthusjax.core.base import BasePopulation

print(f"JAX Version: {jax.__version__}")

JAX Version: 0.8.0


# Tutorial: Custom Engine Development with MalthusJAX

This tutorial demonstrates MalthusJAX's **"Full Access" extensible architecture** for evolutionary engines. Unlike rigid GA libraries that hide internal state, MalthusJAX gives you complete control over the evolutionary process.

## What We'll Cover
1. Run the standard `GeneticEngine` (baseline)
2. Use the built-in `DiversityAwareEngine` (diversity-aware selection)
3. Understand the **Full Access component design**
4. Learn extension patterns for creating custom engines

## The "Full Access" Architecture

MalthusJAX engines follow a **Template Method** pattern where every internal component method receives the **complete evolution state**:

```python
def _select_parents(self, key, state, params) -> BasePopulation:
    # FULL ACCESS TO:
    # ✓ state.population       - Current population with fitness values
    # ✓ state.generation       - Current generation number
    # ✓ state.stagnation_counter - Generations without improvement
    # ✓ state.best_fitness     - Best fitness found so far
    # ✓ state.best_genome      - Best genome found so far
    # ✓ params.pop_size        - Population size
    # ✓ params.elitism         - Number of elites to preserve
    pass
```

### Why This Matters

**Rigid Design (Old Way)**:
```python
def _select_parents(self, key, fitness_values):
    # ✗ No access to population structure
    # ✗ No access to generation count
    # ✗ Cannot compute auxiliary metrics (diversity, novelty)
    # ✗ Cannot implement adaptive algorithms
```

**Full Access Design (MalthusJAX)**:
```python
def _select_parents(self, key, state, params):
    # ✓ Compute diversity metrics: state.population.distance_matrix()
    # ✓ Implement adaptive logic: if state.stagnation_counter > 10: ...
    # ✓ Multi-objective selection: combine fitness + diversity + novelty
    # ✓ Context-aware decisions: adjust selection pressure by generation
```

### Key Extension Points
All component methods use the **unified signature** `(key, state, params)`:
- **`_select_parents(key, state, params)`**: Controls parent selection logic
- **`_select_elites(key, state, params)`**: Controls elite preservation strategy
- **`_create_offspring(key, parents, state, params)`**: Controls variation operators
- **`_merge_and_evaluate(key, elites, offspring, state, params)`**: Controls population assembly

All engines are:
- **JIT-compilable** using `@struct.dataclass` from Flax
- **Immutable** pytrees compatible with `jax.lax.scan`
- **Modular** - operators are injected dependencies

In [9]:
# Setup: Problem Configuration
genome_config = mjx.BinaryGenomeConfig(length=100)
evaluator = mjx.BinarySumEvaluator(config=mjx.BinarySumConfig(maximize=True), data=None)

# Genetic Operators - Now work with direct imports!
selection = mjx.TournamentSelection(num_selections=100, tournament_size=3)
crossover = mjx.UniformCrossover(crossover_rate=0.8)
mutation = mjx.BitFlipMutation(mutation_rate=0.01)

# Engine Parameters
params = mjx.GeneticEngineParams(
    pop_size=100,
    num_generations=50,
    elitism=5
)

print("✓ Problem and operators configured")
print(f"  Genome length: {genome_config.length}")
print(f"  Population: {params.pop_size}, Generations: {params.num_generations}, Elites: {params.elitism}")

✓ Problem and operators configured
  Genome length: 100
  Population: 100, Generations: 50, Elites: 5


## Experiment 1: Standard Genetic Engine (Baseline)

The `GeneticEngine` implements a classic generational GA:
- **Selection**: Tournament selection based purely on fitness
- **Variation**: Uniform crossover + bit-flip mutation
- **Survival**: Elitism preserves top N individuals

In [10]:
# Instantiate Standard Engine
standard_engine = mjx.GeneticEngine(
    genome_config=genome_config,
    evaluator=evaluator,
    selection=selection,
    crossover=crossover,
    mutation=mutation
)

# Initialize and Run
key = jar.PRNGKey(42)
state = standard_engine.init_state(key, params)

print(f"Initial Best Fitness: {state.best_fitness:.4f}")

# Run with JIT compilation
final_state, history, elapsed = standard_engine.run(
    state, 
    params, 
    time_it=True, 
    compile=True,
    verbose=False
)

print(f"✓ Standard GA completed in {elapsed:.3f}s")
print(f"  Final Best Fitness: {final_state.best_fitness:.4f}")
print(f"  Stagnation Counter: {final_state.stagnation_counter}")

Initial Best Fitness: 65.0000
✓ Standard GA completed in 0.623s
  Final Best Fitness: 100.0000
  Stagnation Counter: 21
✓ Standard GA completed in 0.623s
  Final Best Fitness: 100.0000
  Stagnation Counter: 21


## Experiment 2: Diversity-Aware Engine

The `DiversityAwareEngine` demonstrates the power of **Full Access architecture** by overriding selection methods to balance:
- **Fitness** (exploitation): Reward higher-performing individuals
- **Diversity** (exploration): Reward isolated/unique individuals

### Implementation: How Full Access Enables Diversity-Aware Selection

**Key Insight**: The old design couldn't do this! Without access to `state.population`, you couldn't compute the distance matrix needed for diversity metrics.

#### Step 1: Override `_select_parents` with Full State Access

```python
@struct.dataclass
class DiversityAwareEngine(GeneticEngine):
    diversity_weight: float = struct.field(default=0.3, pytree_node=False)
    
    def _select_parents(self, key, state, params) -> BasePopulation:
        # FULL ACCESS: Extract population from state
        population = state.population
        
        # Compute diversity metrics using distance matrix
        # (This requires accessing population structure!)
        crowding_scores = self._compute_crowding_scores(population)
        
        # Combine fitness and diversity
        diversity_fitness = self._compute_diversity_fitness(
            population.fitness,  # Fitness values
            crowding_scores      # Diversity scores
        )
        
        # Use standard selection operator with combined scores
        indices = self.selection(key, diversity_fitness).flatten()
        
        # Return selected parents
        return population[indices]
```

#### Step 2: Compute Crowding Distance

```python
def _compute_crowding_scores(self, population):
    # Compute population distance matrix
    # (requires Full Access to population object!)
    dist_matrix = population.distance_matrix(metric="hamming")
    
    # Crowding = average distance to all others (higher = more isolated)
    dist_matrix = dist_matrix.at[jnp.diag_indices_from(dist_matrix)].set(0.0)
    crowding_scores = jnp.sum(dist_matrix, axis=1)
    
    return crowding_scores
```

#### Step 3: Combine Fitness and Diversity

```python
def _compute_diversity_fitness(self, fitness, crowding):
    # Normalize both metrics to [0, 1]
    fitness_norm = (fitness - jnp.min(fitness)) / (jnp.max(fitness) - jnp.min(fitness) + 1e-8)
    crowding_norm = (crowding - jnp.min(crowding)) / (jnp.max(crowding) - jnp.min(crowding) + 1e-8)
    
    # Weighted combination
    combined = (1 - self.diversity_weight) * fitness_norm + self.diversity_weight * crowding_norm
    
    return combined
```

### Why This Wasn't Possible Before

**Old Design** (without Full Access):
```python
def _select_parents(self, key, fitness_values):
    # ✗ Only have fitness values - no population structure
    # ✗ Cannot compute distance_matrix()
    # ✗ Cannot access state.generation or state.stagnation_counter
    # ✗ Forced to use pure fitness-based selection
    indices = self.selection(key, fitness_values)
    return indices
```

**New Design** (with Full Access):
```python
def _select_parents(self, key, state, params):
    # ✓ Full access to state.population
    # ✓ Can compute auxiliary metrics (diversity, novelty, age)
    # ✓ Can implement adaptive logic based on state.stagnation_counter
    # ✓ Can combine multiple objectives
    population = state.population
    diversity = population.distance_matrix(metric="hamming")
    # ... custom selection logic ...
    return population[indices]
```

### Additional Override: Elite Selection

The engine also overrides `_select_elites` to maintain diverse elites:

```python
def _select_elites(self, key, state, params):
    n_elites = params.elitism
    population = state.population
    
    # Split elites: 70% fitness, 30% diversity
    n_fitness_elites = int(n_elites * 0.7)
    n_diversity_elites = n_elites - n_fitness_elites
    
    # Select top fitness elites
    _, fitness_elite_indices = jax.lax.top_k(population.fitness, n_fitness_elites)
    
    # Select top diversity elites
    crowding_scores = self._compute_crowding_scores(population)
    _, diversity_elite_indices = jax.lax.top_k(crowding_scores, n_diversity_elites)
    
    # Combine and return elite genes
    elite_indices = jnp.concatenate([fitness_elite_indices, diversity_elite_indices])
    return population[elite_indices].genes
```

**Parameters**:
- `diversity_weight=0.3` (default): 70% fitness, 30% diversity
- `distance_metric="hamming"` (default): Distance measure for binary genomes

Let's run it!

In [11]:
# Instantiate Diversity-Aware Engine
diversity_engine = mjx.DiversityAwareEngine(
    genome_config=genome_config,
    evaluator=evaluator,
    selection=selection,
    crossover=crossover,
    mutation=mutation,
    diversity_weight=0.5,  # 50-50 balance between fitness and diversity
    distance_metric="hamming"
)

# Initialize and Run
key = jar.PRNGKey(42)
state = diversity_engine.init_state(key, params)

print(f"Initial Best Fitness: {state.best_fitness:.4f}")

# Run with JIT compilation
final_state_div, history_div, elapsed_div = diversity_engine.run(
    state,
    params,
    time_it=True,
    compile=True,
    verbose=False
)

print(f"✓ Diversity-Aware GA completed in {elapsed_div:.3f}s")
print(f"  Final Best Fitness: {final_state_div.best_fitness:.4f}")
print(f"  Stagnation Counter: {final_state_div.stagnation_counter}")

Initial Best Fitness: 65.0000
✓ Diversity-Aware GA completed in 0.505s
  Final Best Fitness: 96.0000
  Stagnation Counter: 4
✓ Diversity-Aware GA completed in 0.505s
  Final Best Fitness: 96.0000
  Stagnation Counter: 4


## Diversity Analysis

Let's quantify the diversity maintained by each engine using the distance matrix.

In [12]:
def compute_population_diversity(population: BasePopulation) -> float:
    """Compute average Hamming distance in population (excluding self)."""
    dist_matrix = population.distance_matrix(metric="hamming")
    # Zero diagonal
    dist_matrix = dist_matrix.at[jnp.diag_indices_from(dist_matrix)].set(0.0)
    # Average distance
    return float(jnp.mean(dist_matrix))

# Compute diversity metrics
standard_diversity = compute_population_diversity(final_state.population)
diversity_aware_diversity = compute_population_diversity(final_state_div.population)

print("=" * 60)
print("DIVERSITY COMPARISON")
print("=" * 60)
print(f"Standard GA:")
print(f"  Final Diversity: {standard_diversity:.4f}")
print(f"  Best Fitness:    {final_state.best_fitness:.4f}")
print()
print(f"Diversity-Aware GA:")
print(f"  Final Diversity: {diversity_aware_diversity:.4f}")
print(f"  Best Fitness:    {final_state_div.best_fitness:.4f}")
print()
print(f"Diversity Improvement: {(diversity_aware_diversity/standard_diversity - 1)*100:.1f}%")
print(f"Theoretical Max Diversity: ~{genome_config.length * 0.5:.1f} (random population)")
print("=" * 60)

DIVERSITY COMPARISON
Standard GA:
  Final Diversity: 3.1520
  Best Fitness:    100.0000

Diversity-Aware GA:
  Final Diversity: 32.7660
  Best Fitness:    96.0000

Diversity Improvement: 939.5%
Theoretical Max Diversity: ~50.0 (random population)


## How to Create Your Own Custom Engine

The MalthusJAX **"Full Access" architecture** makes it easy to create custom engines. Here's the complete recipe:

### 1. Inherit from `GeneticEngine`

```python
from flax import struct
from malthusjax.engine import GeneticEngine, GeneticEngineParams
from malthusjax.engine.base import AbstractEvolutionState
import jax.numpy as jnp

@struct.dataclass
class MyCustomEngine(GeneticEngine):
    # Add custom parameters as fields
    # pytree_node=False means this is static config, not trainable
    my_param: float = struct.field(default=0.5, pytree_node=False)
```

### 2. Override Component Methods with Full Access Signatures

**CRITICAL: All methods receive (key, state, params)**

```python
def _select_parents(
    self, 
    key: jax.Array,                # RNG key for randomness
    state: AbstractEvolutionState, # FULL STATE ACCESS
    params: GeneticEngineParams    # Engine configuration
) -> BasePopulation:               # Return selected parents
    """
    Override this to customize parent selection.
    
    Full Access Enables:
    - state.population: Current population with fitness
    - state.generation: Current generation number
    - state.stagnation_counter: Generations without improvement
    - state.best_fitness: Best fitness so far
    - state.best_genome: Best genome so far
    """
    population = state.population
    
    # Your custom selection logic here...
    # Example: Access population methods
    diversity = population.distance_matrix(metric="hamming")
    
    # Example: Adaptive logic based on stagnation
    if state.stagnation_counter > 10:
        # Increase selection pressure
        selection_pressure = 2.0
    else:
        selection_pressure = 1.0
    
    # Use standard operators with custom logic
    indices = self.selection(key, population.fitness).flatten()
    return population[indices]

def _select_elites(
    self,
    key: jax.Array,
    state: AbstractEvolutionState,
    params: GeneticEngineParams
) -> jax.Array:  # Return elite GENES only (not population)
    """
    Override this to customize elite preservation.
    
    Return genes directly, not a population object.
    """
    n_elites = params.elitism
    population = state.population
    
    # Your custom elite selection logic...
    _, elite_indices = jax.lax.top_k(population.fitness, n_elites)
    
    return population[elite_indices].genes
```

### 3. Real-World Example: Adaptive Mutation Engine

```python
@struct.dataclass
class AdaptiveMutationEngine(GeneticEngine):
    base_mutation_rate: float = struct.field(default=0.01, pytree_node=False)
    stagnation_threshold: int = struct.field(default=5, pytree_node=False)
    
    def _create_offspring(self, key, parents, state, params):
        """
        Increase mutation rate when evolution stagnates.
        
        This is ONLY possible with Full Access to state!
        """
        # Access stagnation counter from state
        stagnation = state.stagnation_counter
        
        # Adaptive mutation rate scheduling
        if stagnation > self.stagnation_threshold:
            # Evolution is stagnating - increase exploration
            adaptive_rate = self.base_mutation_rate * (1 + 0.5 * stagnation)
            adaptive_rate = jnp.minimum(adaptive_rate, 0.1)  # Cap at 10%
        else:
            adaptive_rate = self.base_mutation_rate
        
        # Note: In practice, you'd modify the mutation operator's rate
        # This is a conceptual example showing state access
        
        # Continue with parent implementation
        return super()._create_offspring(key, parents, state, params)
```

### 4. Real-World Example: Age-Layered Selection

```python
@struct.dataclass
class AgeLayeredEngine(GeneticEngine):
    max_age: int = struct.field(default=10, pytree_node=False)
    
    def _select_elites(self, key, state, params):
        """
        Preserve only young, fit individuals.
        
        Requires custom state with age tracking.
        """
        # You would extend AbstractEvolutionState to include ages:
        # @struct.dataclass
        # class AgeTrackedState(AbstractEvolutionState):
        #     ages: jax.Array
        
        population = state.population
        n_elites = params.elitism
        
        # Mask out old individuals
        young_mask = state.ages < self.max_age  # Requires custom state!
        fitness_masked = jnp.where(young_mask, population.fitness, -jnp.inf)
        
        # Select best young individuals
        _, elite_indices = jax.lax.top_k(fitness_masked, n_elites)
        
        return population[elite_indices].genes
```

### 5. Real-World Example: Novelty Search

```python
@struct.dataclass
class NoveltySearchEngine(GeneticEngine):
    k_nearest: int = struct.field(default=15, pytree_node=False)
    
    def _select_parents(self, key, state, params):
        """
        Select based on behavioral novelty instead of fitness.
        
        Full Access enables computing novelty from population.
        """
        population = state.population
        
        # Compute behavior distance matrix
        behavior_dist = population.distance_matrix(metric="euclidean")
        
        # Novelty score = average distance to k-nearest neighbors
        sorted_dists = jnp.sort(behavior_dist, axis=1)
        novelty_scores = jnp.mean(sorted_dists[:, 1:self.k_nearest+1], axis=1)
        
        # Select parents based on novelty (not fitness!)
        indices = self.selection(key, novelty_scores).flatten()
        
        return population[indices]
```

## Key Design Principles

1. **Full Access Signature**: Always use `(key, state, params)` for component methods
2. **Immutability**: Use `@struct.dataclass` and `pytree_node=False` for configs
3. **Pure Functions**: All methods must be side-effect free for JIT compilation
4. **Return Types**: 
   - `_select_parents` → `BasePopulation` (population object)
   - `_select_elites` → `jax.Array` (genes only, not population)
   - `_create_offspring` → `jax.Array` (genes only)
5. **Flatten Indices**: Always call `.flatten()` on selection indices to avoid dimension errors
6. **State Access**: Leverage `state.population`, `state.generation`, `state.stagnation_counter` for adaptive logic

## Testing Your Engine

```python
def test_custom_engine():
    engine = MyCustomEngine(
        genome_config=mjx.BinaryGenomeConfig(length=10),
        evaluator=mjx.BinarySumEvaluator(...),
        selection=mjx.TournamentSelection(...),
        crossover=mjx.UniformCrossover(...),
        mutation=mjx.BitFlipMutation(...),
        my_param=0.7
    )
    
    params = mjx.GeneticEngineParams(pop_size=20, num_generations=10, elitism=2)
    key = jar.PRNGKey(42)
    
    # Test initialization
    state = engine.init_state(key, params)
    assert state.population is not None
    
    # Test component methods
    key, k_test = jar.split(key)
    parents = engine._select_parents(k_test, state, params)
    assert len(parents) == params.pop_size
    
    key, k_test = jar.split(key)
    elites = engine._select_elites(k_test, state, params)
    assert jax.tree_util.tree_leaves(elites)[0].shape[0] == params.elitism
    
    # Test full run
    final_state, history, elapsed = engine.run(state, params, compile=True)
    assert final_state.generation == params.num_generations
```

## Advanced: Custom Evolution State

For tracking additional metrics (like age, species, archive), subclass `AbstractEvolutionState`:

```python
from malthusjax.engine.base import AbstractEvolutionState

@struct.dataclass
class AgeTrackedState(AbstractEvolutionState):
    ages: jax.Array  # Additional field for tracking genome ages
    
# Then override init_state() in your engine to initialize ages
```

## What Makes Full Access Powerful

| Without Full Access | With Full Access (MalthusJAX) |
|---------------------|-------------------------------|
| ✗ Only see fitness values | ✓ Access complete population structure |
| ✗ Cannot compute diversity | ✓ Compute distance matrices, novelty, clustering |
| ✗ No adaptive algorithms | ✓ Adjust operators based on stagnation, generation |
| ✗ Single-objective only | ✓ Multi-objective, quality-diversity, MAP-Elites |
| ✗ Must rewrite evolution loop | ✓ Override only what you need |

## Conclusion

The **DiversityAwareEngine** source code (`src/malthusjax/engine/diversity_engine.py`) provides a complete, tested example of:
- Overriding `_select_parents` and `_select_elites` with Full Access signatures
- Accessing `state.population` to compute auxiliary metrics (crowding distance)
- Combining multiple objectives (fitness + diversity)
- Maintaining JIT compatibility with `@struct.dataclass`

**Key Takeaway**: The Full Access architecture transforms MalthusJAX from a rigid GA library into a **flexible evolutionary computation platform** where you can rapidly prototype novel algorithms (Novelty Search, MAP-Elites, NSGA-II, Age-Layered Population) without sacrificing performance.

Use the `DiversityAwareEngine` as your reference template!